# הערכת סנכרון — השוואה משולשת (PLAN.md שלב 2)

משווה שלוש שיטות על אותה הקלטה מול ground truth שאתה מסמן:
**baseline** (חלוקה לפי אורך) / **MMS טהור** (המסלול הישן) / **היברידי** (תמלול+עוגנים+חלונות).

זרימה: תא 1 פרמטרים ← Run all ← תא 4 יוצר קובץ גבולות לתיקון ידני ←
מתקנים את הקובץ (מהטלפון, בעורך Drive) ← מריצים שוב את תא 5 בלבד.

In [ ]:
#@title תא 1 — פרמטרים { display-mode: "form" }
BOOK = "בראשית"  #@param ["בראשית", "שמות", "ויקרא", "במדבר", "דברים"]
CHAPTER = 1        #@param {type:"integer"}
VERSE_START = 0    #@param {type:"integer"}
VERSE_END = 0      #@param {type:"integer"}
CHAPTER_END = 0    #@param {type:"integer"}
AUDIO_PATH = "/content/drive/MyDrive/kriat/recording.m4a"  #@param {type:"string"}
MAX_WORDS = 4      #@param {type:"integer"}
BRANCH = "main"

In [ ]:
#@title תא 2 — התקנה + קוד + Drive
import os, subprocess, sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "אין (איטי)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "uroman", "faster-whisper", "silero-vad"], check=True)
if os.path.exists("Kriat-Hatora"):
    subprocess.run(["git", "-C", "Kriat-Hatora", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/michaelnahmias1/Kriat-Hatora.git"], check=True)
sys.path.insert(0, "Kriat-Hatora/src")
from google.colab import drive
drive.mount("/content/drive")
assert os.path.exists(AUDIO_PATH), f"הקובץ לא נמצא: {AUDIO_PATH}"
print("✅ מוכן")

In [ ]:
#@title תא 3 — הרצת שלוש השיטות על אותה הקלטה
from pathlib import Path
from aligner import pipeline as pl
from aligner import hybrid, evaluate

book = pl.TORAH_BOOKS.get(BOOK, BOOK)
version = pl.find_mam_version(book)
ref = pl.build_ref(book, CHAPTER, VERSE_START or None,
                   CHAPTER_END or None, VERSE_END or None)
verses = pl.fetch_verses(ref, version)
segments = pl.verses_to_segments(verses, MAX_WORDS)
waveform = pl.load_audio_16k_mono(AUDIO_PATH)
audio_dur = waveform.size(1) / 16000
flat_voc = [w for s in segments for w in s["align_vocalized"]]

methods = {}
methods["baseline חלוקה-שווה"] = evaluate.equal_split_baseline(segments, audio_dur)

print("MMS טהור (המסלול הישן)…")
try:
    spans = pl.align_words(waveform, pl.romanize_words(flat_voc))
    starts_mms, _ = pl.segment_bounds(segments, spans)
    methods["MMS טהור"] = starts_mms
except Exception as e:
    print(f"⚠️ נכשל: {e}")

print("היברידי…")
out_srt = str(Path(AUDIO_PATH).with_suffix(".eval.srt"))
hybrid.run_hybrid(book=BOOK, chapter=CHAPTER, audio_path=AUDIO_PATH,
                  out_srt_path=out_srt, verse_start=VERSE_START or None,
                  chapter_end=CHAPTER_END or None, verse_end=VERSE_END or None,
                  max_words=MAX_WORDS)
# קריאת התחילות מה-SRT שנוצר
import re
ts = re.findall(r"(\d+):(\d+):(\d+),(\d+) -->",
                Path(out_srt).read_text(encoding="utf-8-sig"))
methods["היברידי"] = [int(h)*3600 + int(m)*60 + int(s) + int(ms)/1000
                      for h, m, s, ms in ts]
print("✅ שלוש השיטות רצו")

In [ ]:
#@title תא 4 — יצירת קובץ ground truth לתיקון ידני
from pathlib import Path
GT_PATH = str(Path(AUDIO_PATH).with_suffix(".ground_truth.txt"))
if not Path(GT_PATH).exists():
    lines = ["# תקן כל שורה לזמן ההתחלה האמיתי (בשניות) של המקטע.",
             "# אפשר לנגן את ההקלטה בתא הבא כדי לדייק. אל תמחק/תוסיף שורות."]
    for t, seg in zip(methods["היברידי"], segments):
        lines.append(f"{t:8.2f}  # פסוק {seg['verse']}: {seg['display']}")
    Path(GT_PATH).write_text("\n".join(lines), encoding="utf-8")
    print(f"נוצר: {GT_PATH}")
    print("פתח את הקובץ (עורך Drive מהטלפון), תקן זמנים שגויים, ואז הרץ את תא 5.")
else:
    print(f"קיים כבר: {GT_PATH} — מדלג (מחק אותו כדי להתחיל מחדש)")

# נגן לבדיקה נקודתית: שנה את SECOND ונגן ±2 שניות סביבו
SECOND = 0.0  #@param {type:"number"}
if SECOND > 0:
    from IPython.display import Audio, display
    s0 = max(0, int((SECOND - 2) * 16000)); s1 = int((SECOND + 2) * 16000)
    display(Audio(waveform[0, s0:s1].numpy(), rate=16000))

In [ ]:
#@title תא 5 — טבלת ההשוואה מול ה-ground truth המתוקן
from pathlib import Path
from aligner import evaluate

truth = []
for line in Path(GT_PATH).read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if line:
        truth.append(float(line))
assert len(truth) == len(segments), (
    f"{len(truth)} זמנים בקובץ מול {len(segments)} מקטעים — אל תמחק/תוסיף שורות")

print(evaluate.comparison_table(methods, truth))